# Model Ensembling & Stacking - PS-S06E08

This notebook ensembles and stacks our three baseline models (XGBoost, CatBoost, and LightGBM) to make final predictions for **Playground Series – Season 6, Episode 8: Predicting Smartphone Addiction**.

We explore multiple ensembling strategies:
1. **Simple Probability Average**
2. **Rank Average**
3. **Optimized Weighted Probability Average** (using Nelder-Mead simplex search)
4. **Meta-Model Stacking** (using `RidgeCV` on out-of-fold predictions)

## 1. Setup & Imports

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import RidgeCV, LinearRegression, LogisticRegression
from sklearn.model_selection import StratifiedKFold

from ps_s06e08_experiment_setup import ExperimentSetup
from ps_s06e08_model_visualizer import ModelVisualizer

warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
setup = ExperimentSetup(
    model_name='Ensemble',
    use_gpu=False,
    perform_rfe=False,
    perform_optuna_tuning=False
)

seed = setup.set_seeds()
setup.configure_pandas()
setup.suppress_warnings()

TARGET = 'addicted_label'

## 2. Load Base Model Predictions

We load out-of-fold (OOF) validation predictions and test-set predictions for all three models.

In [ ]:
pred_dir = Path('predictions')

models = {
    'XGBoost': 'xgb',
    'LightGBM': 'lgb',
    'CatBoost': 'cb'
}

oofs = {}
tests = {}

for name, prefix in models.items():
    oof_path = pred_dir / f'{prefix}_oof_probs.csv'
    test_path = pred_dir / f'{prefix}_test_probs.csv'
    
    oofs[name] = pd.read_csv(oof_path)
    tests[name] = pd.read_csv(test_path)
    
    print(f"Loaded {name} OOF: {oofs[name].shape}, Test: {tests[name].shape}")

In [ ]:
# Align IDs and retrieve target
first_name = list(models.keys())[0]
train_ids = oofs[first_name]['id'].copy()
test_ids = tests[first_name]['id'].copy()
y = oofs[first_name]['target'].copy()

for name in models:
    assert (oofs[name]['id'] == train_ids).all(), f"OOF IDs do not match for {name}!"
    assert (oofs[name]['target'] == y).all(), f"OOF Targets do not match for {name}!"
    assert (tests[name]['id'] == test_ids).all(), f"Test IDs do not match for {name}!"

X_oof = np.column_stack([oofs[name]['prob_1'] for name in models])
X_test = np.column_stack([tests[name]['prob_1'] for name in models])

print(f"Stacking arrays created.")
print(f"OOF matrix shape: {X_oof.shape}")
print(f"Test matrix shape: {X_test.shape}")

## 3. Correlation Analysis & Single Model Scores

In [ ]:
print("Single Model OOF ROC AUC scores:")
single_scores = {}
for i, name in enumerate(models):
    score = roc_auc_score(y, X_oof[:, i])
    single_scores[name] = score
    print(f"  {name}: {score:.6f}")

In [ ]:
# Plot Pearson correlation between predictions
corr_df = pd.DataFrame(X_oof, columns=list(models.keys()))
plt.figure(figsize=(6, 5))
sns.heatmap(corr_df.corr(method='pearson'), annot=True, cmap='coolwarm', fmt='.4f', vmin=0.8, vmax=1.0)
plt.title('Prediction Correlation Matrix (Pearson)')
plt.show()

## 4. Simple Probability Average & Rank Average

In [ ]:
avg_oof = X_oof.mean(axis=1)
avg_score = roc_auc_score(y, avg_oof)
print(f"Simple Probability Average OOF ROC AUC: {avg_score:.6f}")

In [ ]:
# Convert predictions to percentile ranks before averaging to mitigate calibration offsets
rank_oof = np.zeros_like(X_oof)
for i in range(X_oof.shape[1]):
    rank_oof[:, i] = rankdata(X_oof[:, i]) / len(y)
    
rank_avg_oof = rank_oof.mean(axis=1)
rank_avg_score = roc_auc_score(y, rank_avg_oof)
print(f"Rank Average OOF ROC AUC: {rank_avg_score:.6f}")

## 5. Optimized Weighted Probability Average

Finds the weight mixture that directly maximizes OOF ROC AUC score.

In [ ]:
def loss_func(weights):
    w = weights / np.sum(weights)
    blend = np.dot(X_oof, w)
    return -roc_auc_score(y, blend)

# Nelder-Mead optimization starting with equal weights
res = minimize(loss_func, [1/3]*3, method='Nelder-Mead', bounds=[(0, 1)]*3)
opt_weights = res.x / np.sum(res.x)
opt_blend_oof = np.dot(X_oof, opt_weights)
opt_score = roc_auc_score(y, opt_blend_oof)

print(f"Optimized weights:")
for name, weight in zip(models.keys(), opt_weights):
    print(f"  {name}: {weight:.4f}")
print(f"Optimized Weighted Average OOF ROC AUC: {opt_score:.6f}")

## 6. RidgeCV Stacking

We train a meta-regressor (`RidgeCV`) using out-of-fold cross-validation to combine probabilities optimally while regularizing coefficients.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
stack_oof = np.zeros(len(y))

for train_idx, val_idx in skf.split(X_oof, y):
    meta_model = RidgeCV(alphas=np.logspace(-3, 3, 15))
    meta_model.fit(X_oof[train_idx], y[train_idx])
    stack_oof[val_idx] = meta_model.predict(X_oof[val_idx])

stack_score = roc_auc_score(y, stack_oof)
print(f"RidgeCV Stacking OOF ROC AUC: {stack_score:.6f}")

## 7. Model Performance Comparison

In [ ]:
comparison = {
    'Model/Strategy': list(models.keys()) + [
        'Simple Average',
        'Rank Average',
        'Optimized Weighted Average',
        'RidgeCV Stacking'
    ],
    'OOF ROC AUC': [
        single_scores['XGBoost'],
        single_scores['LightGBM'],
        single_scores['CatBoost'],
        avg_score,
        rank_avg_score,
        opt_score,
        stack_score
    ]
}

comp_df = pd.DataFrame(comparison).sort_values('OOF ROC AUC', ascending=False)
comp_df

## 8. Final Meta-Model Training & Submission

Fits the meta-model on all OOF predictions to learn the final coefficients, then applies them to test predictions.

In [ ]:
final_meta = RidgeCV(alphas=np.logspace(-3, 3, 15))
final_meta.fit(X_oof, y)

print("Final Stacking Coefficients:")
for name, coef in zip(models.keys(), final_meta.coef_):
    print(f"  {name}: {coef:.4f}")
print(f"Intercept: {final_meta.intercept_:.4f}")
print(f"Selected Regularization Alpha: {final_meta.alpha_:.4f}")

In [ ]:
# Apply meta-model to test predictions and clip to [0.0, 1.0] range
final_test_preds = final_meta.predict(X_test)
final_test_preds = np.clip(final_test_preds, 0.0, 1.0)

# Apply meta-model to OOF predictions for visualization
final_oof_preds = final_meta.predict(X_oof)
final_oof_preds = np.clip(final_oof_preds, 0.0, 1.0)

In [ ]:
submission_df = setup.read_dataset('submission')
submission_df['addicted_label'] = final_test_preds
submission_df.to_csv('submission.csv', index=False)
print("Saved stacked submission.csv successfully!")

## 9. Final Ensemble Visualization

In [ ]:
mviz = ModelVisualizer(model_name='RidgeCV Stacking')
mviz.plot_roc_curve(y_true=y, y_score=final_oof_preds)

In [ ]:
final_oof_class = (final_oof_preds > 0.5).astype(int)
mviz.plot_confusion_matrix(
    y_true=y,
    y_pred=final_oof_class,
    classes=['Not Addicted', 'Addicted']
)

In [ ]:
# Save ensemble predictions to cache folder
# Format ensemble probabilities for setup.save_probabilities expects column 0 = 1 - p, column 1 = p
final_oof_probs = np.column_stack([1.0 - final_oof_preds, final_oof_preds])
final_test_probs = np.column_stack([1.0 - final_test_preds, final_test_preds])

setup.save_probabilities(
    'ensemble',
    final_oof_probs,
    train_ids,
    y,
    np.ones(len(y), dtype=bool),
    final_test_probs,
    test_ids
)